In [1]:
import pandas as pd

df = pd.read_csv("../datasets/automobile.csv")
X = df.drop(columns="price")
y = df["price"]

print(f"Rows: {len(df)}")
print(f"Features: {X.shape[1]}")
print(f"Rows with target price: {y.notna().sum()}")

Rows: 201
Features: 25
Rows with target price: 201


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "make", "fuel_type", "aspiration", "num_of_doors", "body_style",
    "drive_wheels", "engine_location", "engine_type", "num_of_cylinders",
    "fuel_system",
]
numeric_features = [column for column in X.columns if column not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

model = Pipeline([
    ("preprocessing", preprocessor),
    ("regressor", ExtraTreesRegressor(
        n_estimators=500,
        max_features=0.9,
        random_state=42,
        n_jobs=-1,
    )),
])

In [3]:
# Cross-validation is performed only on the training split; the test split remains held out.
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(model, X_train, y_train, cv=cv, scoring="r2")

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Training 5-fold CV R²: {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")
print(f"Held-out test R²: {r2_score(y_test, y_pred):.3f}")
print(f"Held-out test MAE: {mean_absolute_error(y_test, y_pred):,.0f} price units")

Training 5-fold CV R²: 0.873 ± 0.026
Held-out test R²: 0.945
Held-out test MAE: 1,565 price units
